In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pylab as plt
import datetime
import scipy.stats
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.colors as cl
import matplotlib.dates as mdates
import matplotlib
import statsmodels.api as sm
from statsmodels.regression.rolling import RollingOLS
import seaborn as sns
import netCDF4 as nc
import xarray as xr
import cartopy.crs as ccrs
from cartopy import feature as cfa
import plotly.io as pio
#pio.renderers.default = 'browser'
pio.renderers.default = 'notebook'

In [ ]:
def decimal_year_to_datetime(decimal_year):
    year = decimal_year.astype(int)
    decimal_part = decimal_year - year
    start_of_year = pd.to_datetime([f'{y}-01-01' for y in year])
    seconds_in_year = (pd.to_datetime([f'{y+1}-01-01' for y in year]) - start_of_year).total_seconds()
    seconds = decimal_part * seconds_in_year
    return start_of_year + pd.to_timedelta(seconds, unit='s')

def datetime_to_decimal_year(datetime_):
    year = datetime_.year
    days = (datetime_.date() - datetime.date(year, 1,1)).days
    res = year + days/365.25
    return res


def FFT_res(x, si):
    n = len(x)
    x_FFT_shifted = np.fft.fftshift(np.fft.fft(x))
    x_FFT_freq_shifted = np.fft.fftshift(np.fft.fftfreq(n, d=si))
    return x_FFT_freq_shifted, x_FFT_shifted


def fft_noramlize(fft_res):
    n = len(fft_res)
    norm_fft = np.abs(fft_res)/(np.max(np.abs(fft_res)))
    return norm_fft

def line_fit(X,Y):
    P = np.polyfit(X,Y,1)
    dx = np.max(X) - np.min(X)
    X_new = np.array([np.min(X)-dx*0.05, np.max(X)+dx*0.05])
    Y_new = np.polyval(P, X_new)
    return X_new, Y_new, P

In [ ]:
window_size = 3
window_size2 = 36


df = pd.read_excel('../../2025-12-20_AgeModel-NewXrayPatch/age_model/interp_20251220.xlsx')
paths = list(df['path_name'].value_counts().index.values)
paths = np.flip(np.sort(paths))
for path in paths:
    df.loc[df['path_name']==path, "path_ma"] = df.loc[df['path_name']==path, 'd13c'].rolling(window_size, center=True).mean()
#df[['d13c_ma','d18o_ma']] = df[['d13c','d18o']].rolling(window_size, center=True).mean()
df_sorted = df.sort_values("age_absolute")
df_sorted['d18o_pdb'] = 0.97001*df_sorted['d18o']-29.99

df_sorted['age_ws'] = df_sorted['age_absolute'].rolling(window_size, center=True).mean()
df_sorted['age_ws2'] = df_sorted['age_absolute'].rolling(window_size2, center=True).mean()

df_sorted[['d13c_ma','d18o_ma','d18o_pdp_ma']] = df_sorted[['d13c','d18o','d18o_pdb']].rolling(window_size, center=True).mean()
df_sorted[['d13c_ma2','d18o_ma2','d18o_pdb_ma2']] = df_sorted[['d13c','d18o','d18o_pdb']].rolling(window_size2, center=True).mean()
df_sorted[['d13c_std','d18o_std', 'd18o_pdb_std']] = df_sorted[['d13c','d18o','d18o_pdb']].rolling(window_size2, center=True).std()
df_sorted['d13c_no_tectonic'] = df_sorted['d13c'] - df_sorted['d13c_ma2'].fillna(0)
df_sorted['d18o_no_tectonic'] = df_sorted['d18o'] - df_sorted['d18o_ma2'].fillna(0)
df_sorted['d18o_pdb_no_tectonic'] = df_sorted['d18o_pdb'] - df_sorted['d18o_pdb_ma2'].fillna(0)
df_sorted[['d13c_notec_ma','d18o_notec_ma','d18o_pdb_notec_ma']] = df_sorted[['d13c_no_tectonic','d18o_no_tectonic','d18o_pdb_no_tectonic']].rolling(window_size, center=True).mean()

#-----------------------RESAMPLE MEAN & STD-----------------------------------------------------

df_sorted['age_dt'] = decimal_year_to_datetime(df_sorted['age_absolute'])
df_resampled_ = df_sorted.set_index('age_dt')
df_resampled_.drop(['Name', 'Mass', 'Amp 44', 'Area 44', 'index','sample','age1','path_name',
       'path_num', 'path_ma', 'age_ws', 'age_ws2',
       'd13c_ma', 'd18o_ma', 'd13c_ma2', 'd18o_ma2', 'd13c_std', 'd18o_std'], axis=1, inplace=True)
df_resampled = df_resampled_.resample('30D').mean()

df_resampled.interpolate(inplace=True)
df_resampled[['d13c_ma','d18o_ma']] = df_resampled[['d13c','d18o']].rolling(window_size, center=True).mean()
df_resampled[['d13c_ma2','d18o_ma2']] = df_resampled[['d13c','d18o']].rolling(window_size2, center=True).mean()
df_resampled[['d13c_std','d18o_std']] = df_resampled[['d13c','d18o']].rolling(window_size2, center=True).std()

#-----------------------------------------------------------------------------------------------


#-----------------------CENTRAL DIFF METHOD-----------------------------------------------------
window = 12
col = 'd18o'

last = df_resampled.rolling(window, center=True).agg(lambda x: x.iloc[-1])
first = df_resampled.rolling(window, center=True).agg(lambda x: x.iloc[0])
df_resampled[col+'_cdm'] = (last[col] - first[col]) / (last['age_absolute'] - first['age_absolute'])

window = 12
col = 'd13c'

last = df_resampled.rolling(window, center=True).agg(lambda x: x.iloc[-1])
first = df_resampled.rolling(window, center=True).agg(lambda x: x.iloc[0])
df_resampled[col+'_cdm'] = (last[col] - first[col]) / (last['age_absolute'] - first['age_absolute'])
#-----------------------------------------------------------------------------------------------


#-----------------------------Moving OSL---------------------------------------
coefs_centered = pd.DataFrame()

# Define multiple windows and target columns
windows = [36, 36]
cols = ['d18o', 'd13c']

for window, col in zip(windows, cols):
    
    # Prepare data (assumes df_resampled contains all variables)
    data = df_resampled[[col, 'age_absolute']].dropna()

    if len(data) < window:
        print(f"Not enough data for column '{col}' with window {window}. Skipping.")
        continue
    
    # Setup endogenous and exogenous variables
    endog = data[col]
    exog = sm.add_constant(data['age_absolute'])

    # Fit rolling OLS
    rols = RollingOLS(endog, exog, window=window)
    rres = rols.fit()

    # Shift result to center and rename columns
    coefs = rres.params.shift(-(window // 2)).dropna()
    coefs = coefs.rename(columns={'const': f'{col}_intercept', 'age_absolute': f'{col}_slope'})

    # Merge into result
    coefs_centered = coefs_centered.join(coefs, how='outer')

# Add shared age column (assumes time index is same)
coefs_centered['age_absolute'] = df_resampled['age_absolute']

# Drop rows with NaNs in all slope columns (optional)
slope_cols = [f"{col}_slope" for col in cols]
coefs_centered.dropna(subset=slope_cols, how='all', inplace=True)
 
#-----------------------------------------------------------------------------------------------


#---------------------------------------FFT-----------------------------------------------------
d13c_fft_freq, d13c_fft = FFT_res(df_resampled['d13c'], 30)
d18o_fft_freq, d18o_fft = FFT_res(df_resampled['d18o'], 30)

df_resampled[['d13c_std','d18o_std']] = df_resampled[['d13c','d18o']].rolling(window=window_size2, center=True).std()
d13c_stds = df_resampled.dropna()['d13c_std']
d18o_stds = df_resampled.dropna()['d18o_std']
d13c_std_fft_freq, d13c_std_fft = FFT_res(d13c_stds, 30)
d18o_std_fft_freq, d18o_std_fft = FFT_res(d18o_stds, 30)
#-----------------------------------------------------------------------------------------------


df_resampled

In [ ]:
df_sorted.iloc[60:90]

In [ ]:
plt.plot(df_sorted['index'].values)

In [ ]:
#df_sorted['sample_index'] = df_sorted['age1'].index.values+1
#sampling_rates =  df_sorted[~df_sorted['age1'].isna()]['sample_index'].diff()[1:]
#sampling_rates = df_sorted[~df_sorted['age1'].isna()].groupby('path_name')['path_num'].diff().dropna()
sampling_rates = df[~df['age1'].isna()].groupby('path_name')['path_num'].diff().dropna()

In [ ]:
mu = np.mean(sampling_rates)
sigma = np.std(sampling_rates)

fig, ax = plt.subplots(1, figsize=(7,5))

count, bins, ignored = ax.hist(sampling_rates, bins=np.arange(1,15), rwidth=0.9, align='left' ,density=True)
ax.plot(bins, 1/(sigma * np.sqrt(2 * np.pi)) *
               np.exp( - (bins - mu)**2 / (2 * sigma**2) ),
         linewidth=2, color='r')
ax.set_xticks(np.arange(1,15))
ax.set_xlim(1,15)

ax.set_title('Sampling Rate Distribution')
ax.set_xlabel('# of Sample per Year')
ax.set_ylabel('Probability')

#np.sum(1/(sigma * np.sqrt(2 * np.pi)) *     np.exp( - (bins - mu)**2 / (2 * sigma**2)))

#fig.savefig('sampling_rate_distribution.pdf')

In [ ]:
fig, axes = plt.subplots(1,2, figsize=(12,4), width_ratios=(2,1), sharey=True, gridspec_kw={'wspace':0})

ax = axes[0]
ax.plot(sampling_rates, '.-')

ax = axes[1]

mu = np.mean(sampling_rates)
sigma = np.std(sampling_rates)
bin_count = 16

count, bins, ignored = ax.hist(sampling_rates, bins=np.arange(1,bin_count), rwidth=0.9, align='left',orientation='horizontal' ,density=True)
ax.plot(1/(sigma*np.sqrt(2*np.pi))*np.exp(-(bins-mu)**2 / (2*sigma**2)),bins,linewidth=2, color='r')
ax.set_yticks(np.arange(1,bin_count+1))
ax.set_ylim(1,bin_count+1)


#fig.suptitle('Sampling Rate Distribution')

axes[1].set_title('Sampling Rate Distribution')
axes[0].set_title('Sampling Rate over Time')
axes[0].set_ylabel('# of Sample per Year')
axes[0].set_xlabel('Sample Index ~ Time')
ax.set_xlabel('Probability')

#fig.savefig('sampling_rate_distribution.pdf')

In [ ]:

growth_rates = (sampling_rates*1.5)/10

# Drill Samples were 1.5 mm ?????

#####################################

fig, axes = plt.subplots(1,2, figsize=(12,4), width_ratios=(2,1), sharey=True, gridspec_kw={'wspace':0})

ax = axes[0]
ax.plot(growth_rates, '.-')
ax.grid()

ax = axes[1]

mu = np.mean(growth_rates)
sigma = np.std(growth_rates)
bin_count = 22

count, bins, ignored = ax.hist(growth_rates, bins=np.arange(1,bin_count)/10, rwidth=0.7, align='left',orientation='horizontal' ,density=True)
ax.plot(1/(sigma*np.sqrt(2*np.pi))*np.exp(-(bins-mu)**2 / (2*sigma**2)),bins,linewidth=2, color='r')
ax.set_yticks(np.arange(1,bin_count+1,2)/10)
ax.set_ylim(1/10,bin_count/10+1/10)
ax.grid()

ax.yaxis.tick_right()
ax.yaxis.set_label_position("right")
ax.set_ylabel("Growth Rate [cm/a]")


#fig.suptitle('Sampling Rate Distribution')

axes[1].set_title('Growth Rate Distribution')
axes[0].set_title('Growth Rate over Sample Age')
axes[0].set_ylabel('Growth Rate [cm/a]')
axes[0].set_xlabel('Sample Index ~ Time')
ax.set_xlabel('Probability')


fig.savefig('sampling_rate_distribution.pdf')

In [ ]:

sampling_rates = df_sorted[~df_sorted['age1'].isna()].groupby('path_name')['path_num'].diff().dropna()

mean = np.mean(sampling_rates)
std_dev = np.std(sampling_rates)

print(f"Mean: {mean}")
print(f"Standard Deviation: {std_dev}")

size = 1000 # Number of samples to generate

# Generate new normally distributed samples
new_data = np.random.normal(loc=mean, scale=std_dev, size=size)

# Optional: Visualize the new distribution as a histogram

fig, ax = plt.subplots(1, figsize=(8,6))

ax.hist(sampling_rates, bins=np.arange(1,15))

sns.kdeplot(new_data, ax=ax) # requires the seaborn library


In [ ]:
age_est = pd.read_csv('export/age_est.csv', header=None)
climate_mean = pd.read_csv('export/climMean.csv', header=None)
ts_abs = pd.read_csv('export/ts_abs.csv', header=None)
y_coral = pd.read_csv('export/y_coral.csv', header=None)

In [ ]:
ts_abs.iloc[:,0]

In [ ]:
df = pd.DataFrame([])
df['d18o'] = y_coral
df['age_est'] = age_est
df

In [ ]:
df_sorted['age_absolute']

In [ ]:

fig = make_subplots(specs=[[{"secondary_y": True}]])

fig.add_trace(go.Scatter(x=df_sorted['age_absolute'], y=df_sorted['d18o'],  mode="lines", marker_color="lightblue",  name='old age model', text=df_sorted['Name']),secondary_y=False)
#fig.add_trace(go.Scatter(x=df_resampled.index.values, y=df_resampled['d18o_ma2'], mode="lines", line_color='navy', name='average'), secondary_y=False)

#fig.add_trace(go.Scatter(x=oras5_t, y=synth_d18o, mode="lines", line_color='#F54927', name='synthetic d18o', line_width=1), secondary_y=True)

fig.add_trace(go.Scatter(x=ts_abs.iloc[:,0],  y=ts_abs.iloc[:,1],  mode="lines", marker_color="blue",   name='new age model', text=df_sorted['Name']),secondary_y=False)


#fig.update_layout(
#    yaxis2=dict(title="Secondary Y-axis Title", overlaying="y", side="right", gridcolor='lightgray', showgrid=True)) # Grid for secondary Y-axis)


fig.update_layout(width=1600, height=600, title_text="d18O old and new age models", template='simple_white')
# Set x-axis title
fig.update_xaxes(title_text="Year")

# Set y-axes titles
fig.update_yaxes(title_text= r'$\delta O ^{18}$', secondary_y=False)
#fig.update_yaxes(title_text="Tide Gauge [mm]", secondary_y=True)

fig.show()
#fig.write_html('d18o_syntetic-d18O-CONSTANT-TEMP.html')
